In [1]:
# ========== ЯЧЕЙКА 1: ИМПОРТЫ, КОНСТАНТЫ И ФИКСАЦИЯ SEED ==========
import requests
import zipfile
import os
import cv2
import numpy as np
import shutil
import random
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

PUBLIC_URL = "https://disk.yandex.ru/d/LUQAjPM_vbnCfg"
OUTPUT_DIR = Path("./dataset")
TEST_DIR = Path("./test_dataset")
TRAIN_DIR = Path("./train_dataset")
ZIP_PATH = OUTPUT_DIR / "dataset.zip"
SLICE_SIZE = 8
STRIDE = 4
TEST_RATIO = 0.1

In [2]:
# ========== ЯЧЕЙКА 2: СКАЧИВАНИЕ И РАСПАКОВКА ДАТАСЕТА ==========
def download_and_extract():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    resp = requests.get("https://cloud-api.yandex.net/v1/disk/public/resources/download", params={"public_key": PUBLIC_URL})
    resp.raise_for_status()
    direct_link = resp.json()["href"]

    print("Downloading...")
    with requests.get(direct_link, stream=True) as r:
        r.raise_for_status()
        with open(ZIP_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(OUTPUT_DIR)

    ZIP_PATH.unlink()
    print(f"Done. Extracted to {OUTPUT_DIR}")

download_and_extract()

Downloading...
Extracting...
Done. Extracted to dataset


In [3]:
# ========== ЯЧЕЙКА 3: РАЗБИЕНИЕ НА TRAIN И TEST (10% 8-кадровых в тест) ==========
def split_train_test():
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    TEST_DIR.mkdir(parents=True, exist_ok=True)

    test_sequences = []
    train_sequences = []

    for class_dir in OUTPUT_DIR.iterdir():
        if not class_dir.is_dir():
            continue

        train_class_dir = TRAIN_DIR / class_dir.name
        test_class_dir = TEST_DIR / class_dir.name
        train_class_dir.mkdir(parents=True, exist_ok=True)
        test_class_dir.mkdir(parents=True, exist_ok=True)

        eight_frame_sequences = []
        other_sequences = []

        for seq_dir in class_dir.iterdir():
            if not seq_dir.is_dir():
                continue

            frames = []
            for ext in ['*.jpg', '*.png', '*.jpeg']:
                frames.extend(sorted(seq_dir.glob(ext)))

            if len(frames) == SLICE_SIZE:
                eight_frame_sequences.append(seq_dir)
            else:
                other_sequences.append(seq_dir)

        n_test = max(1, int(len(eight_frame_sequences) * TEST_RATIO))
        test_selected = random.sample(eight_frame_sequences, n_test)
        train_eight = [seq for seq in eight_frame_sequences if seq not in test_selected]

        for seq_dir in train_eight + other_sequences:
            train_seq_dir = train_class_dir / seq_dir.name
            train_seq_dir.mkdir(exist_ok=True)
            for frame_path in seq_dir.iterdir():
                if frame_path.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                    shutil.copy2(frame_path, train_seq_dir / frame_path.name)
            train_sequences.append((class_dir.name, seq_dir.name))

        for seq_dir in test_selected:
            test_seq_dir = test_class_dir / seq_dir.name
            test_seq_dir.mkdir(exist_ok=True)
            for frame_path in seq_dir.iterdir():
                if frame_path.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                    shutil.copy2(frame_path, test_seq_dir / frame_path.name)
            test_sequences.append((class_dir.name, seq_dir.name))

    print(f"\nSPLIT RESULTS:")
    print(f"="*50)
    print(f"TEST set (10% of {SLICE_SIZE}-frame sequences):")
    test_counts = defaultdict(int)
    for cls, _ in test_sequences:
        test_counts[cls] += 1
    for cls, cnt in test_counts.items():
        print(f"  {cls}: {cnt} sequences")

    print(f"\nTRAIN set (all other sequences):")
    train_counts = defaultdict(int)
    for cls, _ in train_sequences:
        train_counts[cls] += 1
    for cls, cnt in train_counts.items():
        print(f"  {cls}: {cnt} sequences")

    print(f"\nTotal test: {len(test_sequences)} sequences")
    print(f"Total train: {len(train_sequences)} sequences")

    return test_sequences, train_sequences

test_sequences, train_sequences = split_train_test()


SPLIT RESULTS:
TEST set (10% of 8-frame sequences):
  move: 7 sequences
  work: 24 sequences
  inaction: 6 sequences

TRAIN set (all other sequences):
  move: 143 sequences
  work: 318 sequences
  inaction: 84 sequences

Total test: 37 sequences
Total train: 545 sequences


In [4]:
# ========== ЯЧЕЙКА 4: ФУНКЦИИ ДЛЯ ЗАГРУЗКИ И ПОДГОТОВКИ ДАННЫХ ==========
def load_frames_from_folder(folder_path):
    frames = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        frames.extend(sorted(folder_path.glob(ext)))
    if not frames:
        return None
    return [cv2.imread(str(f)) for f in frames if cv2.imread(str(f)) is not None]

def slice_sequence(frames, slice_size=SLICE_SIZE, stride=STRIDE):
    if len(frames) < slice_size:
        return []
    slices = []
    for start in range(0, len(frames) - slice_size + 1, stride):
        slices.append(frames[start:start + slice_size])
    return slices

def prepare_training_data():
    sequences = []
    sequence_labels = []
    sequence_slices = []

    print("\nPreparing training data...")
    for class_dir in TRAIN_DIR.iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        print(f"Processing {class_name}...")

        for seq_dir in class_dir.iterdir():
            if not seq_dir.is_dir():
                continue

            frames = load_frames_from_folder(seq_dir)
            if not frames or len(frames) < SLICE_SIZE:
                continue

            slices = slice_sequence(frames, SLICE_SIZE, STRIDE)
            if slices:
                sequences.append(seq_dir.name)
                sequence_labels.append(class_name)
                sequence_slices.append(slices)

    print(f"\nTraining sequences: {len(sequences)}")
    seq_counts = defaultdict(int)
    for label in sequence_labels:
        seq_counts[label] += 1
    for cls, cnt in seq_counts.items():
        print(f"  {cls}: {cnt} sequences")

    return sequences, sequence_labels, sequence_slices

train_sequences_names, train_labels, train_slices = prepare_training_data()


Preparing training data...
Processing move...
Processing work...
Processing inaction...

Training sequences: 545
  move: 143 sequences
  work: 318 sequences
  inaction: 84 sequences


In [5]:
# ========== ЯЧЕЙКА 5: DATASET И MODEL КЛАССЫ ==========
class VideoDataset(Dataset):
    def __init__(self, sequence_slices, labels, label_encoder, transform=None, fixed_slice_index=None):
        self.sequence_slices = sequence_slices
        self.labels = labels
        self.label_encoder = label_encoder
        self.fixed_slice_index = fixed_slice_index
        self.transform = transform or transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.sequence_slices)

    def __getitem__(self, idx):
        slices = self.sequence_slices[idx]

        if self.fixed_slice_index is not None:
            slice_idx = min(self.fixed_slice_index, len(slices) - 1)
        else:
            slice_idx = np.random.randint(len(slices))

        frames = slices[slice_idx]

        processed = []
        for frame in frames:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_tensor = self.transform(frame_rgb)
            processed.append(frame_tensor)

        video_tensor = torch.stack(processed)
        label = self.label_encoder.transform([self.labels[idx]])[0]
        return video_tensor, torch.tensor(label, dtype=torch.long)

class VideoClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = models.mobilenet_v2(pretrained=True)
        self.cnn.classifier = nn.Identity()
        self.lstm = nn.LSTM(1280, 256, batch_first=True, dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.shape
        x = x.view(batch_size * seq_len, c, h, w)
        features = self.cnn(x)
        features = features.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(features)
        pooled = lstm_out.mean(dim=1)
        return self.classifier(pooled)

In [6]:
# ========== ЯЧЕЙКА 6: ФУНКЦИЯ ДЛЯ ПРЕДСКАЗАНИЯ ПОСЛЕДОВАТЕЛЬНОСТИ (ДЕТЕРМИНИРОВАННАЯ) ==========
def predict_sequence_deterministic(model, slices, device, label_encoder):
    """Предсказание по ВСЕМ слайсам с усреднением вероятностей"""
    model.eval()
    all_probs = []

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    with torch.no_grad():
        for frames in slices:
            processed = []
            for frame in frames:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame_tensor = transform(frame_rgb)
                processed.append(frame_tensor)

            video_tensor = torch.stack(processed).unsqueeze(0).to(device)
            outputs = model(video_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            all_probs.append(probabilities.cpu().numpy()[0])

    avg_probs = np.mean(all_probs, axis=0)
    final_pred = np.argmax(avg_probs)
    return final_pred, avg_probs

In [7]:
# ========== ЯЧЕЙКА 7: ОБУЧЕНИЕ МОДЕЛИ С СОХРАНЕНИЕМ ВСЕХ 5 ФОЛДОВ ==========
def train_model(sequence_labels, sequence_slices):
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(sequence_labels)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accuracies = []
    all_models = []

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(sequence_slices, y_encoded)):
        print(f"\n{'='*50}")
        print(f"FOLD {fold+1}/5")
        print(f"{'='*50}")
        print(f"Train sequences: {len(train_idx)}, Val sequences: {len(val_idx)}")

        train_slices = [sequence_slices[i] for i in train_idx]
        train_labels = [sequence_labels[i] for i in train_idx]
        val_slices = [sequence_slices[i] for i in val_idx]
        val_labels = [sequence_labels[i] for i in val_idx]

        train_dataset = VideoDataset(train_slices, train_labels, label_encoder, fixed_slice_index=None)
        val_dataset = VideoDataset(val_slices, val_labels, label_encoder, fixed_slice_index=0)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, generator=g)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

        model = VideoClassifier(num_classes=len(label_encoder.classes_)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

        best_val_acc = 0
        best_model_state = None
        patience_counter = 0

        for epoch in range(30):
            model.train()
            train_loss = 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            val_preds = []
            val_true = []
            for i, slices in enumerate(val_slices):
                pred, _ = predict_sequence_deterministic(model, slices, device, label_encoder)
                val_preds.append(pred)
                val_true.append(label_encoder.transform([val_labels[i]])[0])

            val_acc = accuracy_score(val_true, val_preds)
            scheduler.step(val_acc)

            print(f"Epoch {epoch+1:2d}: Loss: {train_loss/len(train_loader):.4f}, Val Acc: {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = model.state_dict().copy()
                patience_counter = 0
                torch.save(best_model_state, f'model_fold_{fold+1}.pth')
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print(f"Early stopping")
                    break

        all_models.append(best_model_state)
        fold_accuracies.append(best_val_acc)
        print(f"Fold {fold+1} best accuracy: {best_val_acc:.4f}")

    print(f"\n{'='*50}")
    print(f"CROSS-VALIDATION RESULTS")
    print(f"{'='*50}")
    print(f"Mean CV Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")

    return all_models, label_encoder

all_models, label_encoder = train_model(train_labels, train_slices)


Using device: cuda

FOLD 1/5
Train sequences: 436, Val sequences: 109
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 133MB/s]


Epoch  1: Loss: 0.8912, Val Acc: 0.6972
Epoch  2: Loss: 0.6906, Val Acc: 0.8165
Epoch  3: Loss: 0.5311, Val Acc: 0.8165
Epoch  4: Loss: 0.3834, Val Acc: 0.8349
Epoch  5: Loss: 0.3444, Val Acc: 0.8624
Epoch  6: Loss: 0.3490, Val Acc: 0.8349
Epoch  7: Loss: 0.2456, Val Acc: 0.8624
Epoch  8: Loss: 0.2345, Val Acc: 0.8807
Epoch  9: Loss: 0.1724, Val Acc: 0.8991
Epoch 10: Loss: 0.1385, Val Acc: 0.9083
Epoch 11: Loss: 0.1526, Val Acc: 0.8440
Epoch 12: Loss: 0.0885, Val Acc: 0.8349
Epoch 13: Loss: 0.1106, Val Acc: 0.8624
Epoch 14: Loss: 0.0901, Val Acc: 0.8532
Epoch 15: Loss: 0.1352, Val Acc: 0.8624
Epoch 16: Loss: 0.0889, Val Acc: 0.8532
Epoch 17: Loss: 0.0978, Val Acc: 0.8624
Epoch 18: Loss: 0.0748, Val Acc: 0.8716
Epoch 19: Loss: 0.1046, Val Acc: 0.8716
Epoch 20: Loss: 0.0561, Val Acc: 0.8716
Early stopping
Fold 1 best accuracy: 0.9083

FOLD 2/5
Train sequences: 436, Val sequences: 109
Epoch  1: Loss: 0.8828, Val Acc: 0.7248
Epoch  2: Loss: 0.5990, Val Acc: 0.7431
Epoch  3: Loss: 0.4452, V

In [15]:
# ========== ЯЧЕЙКА 8 (УПРОЩЕННАЯ): ТОЛЬКО SOFT VOTING + TTA ==========
import torch
import torch.nn.functional as F
from scipy.special import softmax
from collections import Counter

def load_test_sequences():
    """Загрузка тестовых последовательностей"""
    test_data = []

    print("\nLoading test sequences...")
    for class_dir in TEST_DIR.iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name

        for seq_dir in class_dir.iterdir():
            if not seq_dir.is_dir():
                continue

            frames = load_frames_from_folder(seq_dir)
            if frames and len(frames) == SLICE_SIZE:
                test_data.append((frames, class_name, seq_dir.name))

    print(f"Loaded {len(test_data)} test sequences")
    return test_data

def predict_single_with_confidence(model, frames, device):
    """Предсказание с возвратом вероятностей"""
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    processed = []
    for frame in frames:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_tensor = transform(frame_rgb)
        processed.append(frame_tensor)

    video_tensor = torch.stack(processed).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(video_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    return predicted.item(), probabilities.detach().cpu().numpy()[0], confidence.item()

def advanced_tta(model, frames, device, num_augmentations=8):
    """Продвинутое TTA с разнообразными аугментациями"""
    all_probs = []
    confidences = []

    # 1. Оригинал
    _, probs, conf = predict_single_with_confidence(model, frames, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 2. Горизонтальное отражение
    flipped = [cv2.flip(f, 1) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, flipped, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 3. Вертикальное отражение
    flipped_v = [cv2.flip(f, 0) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, flipped_v, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 4. Поворот на 90 градусов
    rotated_90 = [cv2.rotate(f, cv2.ROTATE_90_CLOCKWISE) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, rotated_90, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 5. Яркость +30%
    bright = [np.clip(f * 1.3, 0, 255).astype(np.uint8) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, bright, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 6. Яркость -30%
    dark = [np.clip(f * 0.7, 0, 255).astype(np.uint8) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, dark, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 7. Контраст +30%
    contrast_high = [np.clip(128 + 1.3 * (f - 128), 0, 255).astype(np.uint8) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, contrast_high, device)
    all_probs.append(probs)
    confidences.append(conf)

    # 8. Размытие
    blurred = [cv2.GaussianBlur(f, (5, 5), 0) for f in frames]
    _, probs, conf = predict_single_with_confidence(model, blurred, device)
    all_probs.append(probs)
    confidences.append(conf)

    # Взвешенное усреднение с учетом уверенности модели
    conf_weights = np.array(confidences) / np.sum(confidences)
    weighted_avg_probs = np.average(all_probs, weights=conf_weights, axis=0)

    final_pred = np.argmax(weighted_avg_probs)
    final_confidence = np.max(weighted_avg_probs)

    return final_pred, weighted_avg_probs, final_confidence

def evaluate_soft_voting_tta(models, label_encoder, test_data):
    """Только Soft Voting + TTA"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    num_models = len(models)
    class_names = label_encoder.classes_

    true_labels = []
    seq_names = []
    predictions = []
    confidences = []

    # Загрузка моделей
    loaded_models = []
    for i, model_state in enumerate(models):
        model = VideoClassifier(num_classes=len(class_names)).to(device)
        model.load_state_dict(model_state)
        model.eval()
        loaded_models.append(model)

    print("\nProcessing test sequences with Soft Voting + TTA...")

    # Вычисление весов моделей на основе TTA точности на валидации
    # (используем равные веса или можно вычислить на отдельной валидационной выборке)
    model_weights = np.ones(num_models) / num_models

    for frames, true_class, seq_name in test_data:
        true_labels.append(true_class)
        seq_names.append(seq_name)

        all_probs = []
        for model in loaded_models:
            _, probs, _ = advanced_tta(model, frames, device)
            all_probs.append(probs)

        # Взвешенное усреднение
        avg_probs = np.average(all_probs, weights=model_weights, axis=0)
        pred_idx = np.argmax(avg_probs)
        conf = np.max(avg_probs)

        predictions.append(class_names[pred_idx])
        confidences.append(conf)

    true_labels = np.array(true_labels)
    predictions = np.array(predictions)

    # Вывод результатов
    print("\n" + "="*80)
    print("SOFT VOTING + TTA RESULTS")
    print("="*80)

    acc = accuracy_score(true_labels, predictions)
    print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"Mean confidence: {np.mean(confidences):.4f}")
    print(f"High confidence predictions (>0.9): {np.sum(np.array(confidences) > 0.9)}/{len(confidences)}")

    # Детальный отчет
    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)

    for i, (seq_name, true, pred, conf) in enumerate(zip(seq_names, true_labels, predictions, confidences)):
        status = "✓" if true == pred else "✗"
        if i < 10 or true != pred:
            print(f"{status} {seq_name}: true={true}, predicted={pred}, confidence={conf:.3f}")

    # Confusion matrix
    print("\n" + "="*80)
    print("CONFUSION MATRIX")
    print("="*80)

    cm = confusion_matrix(true_labels, predictions, labels=class_names)
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions, target_names=class_names))

    # Анализ ошибок
    errors_by_class = defaultdict(list)
    for seq_name, true, pred, conf in zip(seq_names, true_labels, predictions, confidences):
        if true != pred:
            errors_by_class[true].append((seq_name, pred, conf))

    if errors_by_class:
        print("\n" + "="*80)
        print("ERRORS BY CLASS")
        print("="*80)
        for class_name, errors in errors_by_class.items():
            print(f"\n{class_name}: {len(errors)} errors")
            for seq_name, pred, conf in errors:
                print(f"  - {seq_name} -> {pred} (conf={conf:.3f})")

    return {
        'predictions': predictions,
        'confidences': confidences,
        'true_labels': true_labels,
        'seq_names': seq_names,
        'accuracy': acc
    }

# Запуск оценки
test_data = load_test_sequences()
results = evaluate_soft_voting_tta(all_models, label_encoder, test_data)


Loading test sequences...
Loaded 37 test sequences

Processing test sequences with Soft Voting + TTA...

SOFT VOTING + TTA RESULTS

Accuracy: 0.9189 (91.89%)
Mean confidence: 0.7976
High confidence predictions (>0.9): 13/37

DETAILED RESULTS
✓ 260: true=move, predicted=move, confidence=0.844
✗ 144: true=move, predicted=work, confidence=0.732
✓ 1016: true=move, predicted=move, confidence=0.763
✓ 184: true=move, predicted=move, confidence=0.817
✓ 822: true=move, predicted=move, confidence=0.856
✗ 64: true=move, predicted=work, confidence=0.902
✓ 1026: true=move, predicted=move, confidence=0.600
✓ 50: true=work, predicted=work, confidence=0.918
✓ 4: true=work, predicted=work, confidence=0.897
✓ 30: true=work, predicted=work, confidence=0.922
✗ 692: true=inaction, predicted=work, confidence=0.937

CONFUSION MATRIX

Confusion Matrix:
[[ 5  0  1]
 [ 0  5  2]
 [ 0  0 24]]

Classification Report:
              precision    recall  f1-score   support

    inaction       1.00      0.83      0.9